In [1]:
import requests
import pandas as pd
import time
from typing import List, Dict, Optional

# Configuration
API_BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
REQUEST_TIMEOUT = 30
API_DELAY = 1.0  # Seconds between requests to respect rate limits

# Conditions to search (diverse disease areas for robust training)
SEARCH_CONDITIONS = [
    'cancer', 'breast cancer', 'lung cancer', 
    'diabetes', 'type 2 diabetes',
    'cardiovascular disease', 'heart failure',
    'depression', 'alzheimer disease', 'asthma'
]

# Keywords indicating trial failure due to scientific reasons
FAILURE_INDICATORS = [
    'futility', 'lack of efficacy', 'insufficient efficacy',
    'safety concerns', 'adverse event', 'toxicity',
    'slow enrollment', 'did not meet endpoint'
]

# Keywords indicating administrative (non-scientific) termination
ADMINISTRATIVE_REASONS = [
    'business decision', 'strategic reasons', 'sponsor decision',
    'funding', 'administrative reasons'
]


def fetch_trials(condition: str, max_results: int = 300) -> List[Dict]:
    """
    Fetch clinical trials for a given condition from ClinicalTrials.gov.
    
    Args:
        condition: Disease or condition to search for
        max_results: Maximum number of trials to retrieve
        
    Returns:
        List of trial dictionaries with extracted information
    """
    params = {
        "query.cond": condition,
        "pageSize": min(max_results, 1000),  # API limit is 1000
        "format": "json"
    }
    
    print(f"Fetching {condition} trials...", end=" ")
    
    try:
        response = requests.get(API_BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()
        
        trials = []
        studies = data.get('studies', [])
        
        for study in studies:
            trial_data = extract_trial_data(study)
            if trial_data:
                trials.append(trial_data)
        
        print(f"Retrieved {len(trials)} trials")
        return trials
        
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []
    except Exception as e:
        print(f"Unexpected error: {e}")
        return []


def extract_trial_data(study: Dict) -> Optional[Dict]:
    """
    Extract relevant fields from a clinical trial record.
    
    Filters out trials without sufficient information or clear outcomes.
    
    Args:
        study: Raw study data from ClinicalTrials.gov API
        
    Returns:
        Dictionary with extracted trial information, or None if insufficient data
    """
    try:
        protocol = study.get('protocolSection', {})
        
        # Extract basic identifiers and descriptions
        id_module = protocol.get('identificationModule', {})
        nct_id = id_module.get('nctId', '')
        title = id_module.get('briefTitle', '')
        
        desc_module = protocol.get('descriptionModule', {})
        description = desc_module.get('briefSummary', '')
        
        elig_module = protocol.get('eligibilityModule', {})
        eligibility = elig_module.get('eligibilityCriteria', '')
        
        # Extract trial status information
        status_module = protocol.get('statusModule', {})
        overall_status = status_module.get('overallStatus', '')
        why_stopped = status_module.get('whyStopped', '')
        
        # Determine outcome label
        outcome = classify_outcome(overall_status, why_stopped)
        
        # Only include trials with clear outcomes and sufficient description
        if outcome == -1 or not description or len(description) < 50:
            return None
        
        return {
            'nct_id': nct_id,
            'title': title,
            'description': description,
            'eligibility': eligibility,
            'status': overall_status,
            'why_stopped': why_stopped,
            'outcome': outcome
        }
        
    except Exception as e:
        # Skip malformed records silently
        return None


def classify_outcome(status: str, why_stopped: str) -> int:
    """
    Classify trial outcome based on completion status and termination reason.
    
    Args:
        status: Overall trial status (e.g., 'COMPLETED', 'TERMINATED')
        why_stopped: Reason for early termination (if applicable)
        
    Returns:
        1 for success, 0 for failure, -1 for unclear/administrative
    """
    status = status.upper()
    why_stopped_lower = why_stopped.lower() if why_stopped else ""
    
    # Completed trials are considered successful
    if status == 'COMPLETED':
        return 1
    
    # Terminated or withdrawn trials need further classification
    if status in ['TERMINATED', 'WITHDRAWN', 'SUSPENDED']:
        # Check if termination was for administrative reasons (not a true failure)
        if any(reason in why_stopped_lower for reason in ADMINISTRATIVE_REASONS):
            return -1
        
        # Check if termination was for scientific failure
        if any(indicator in why_stopped_lower for indicator in FAILURE_INDICATORS):
            return 0
        
        # If stopped with explanation but unclear reason, likely a failure
        if why_stopped and len(why_stopped) > 10:
            return 0
    
    # Unclear outcome for other statuses
    return -1


def main():
    """Main execution: download trials and save to CSV."""
    print("=" * 70)
    print("Clinical Trials Data Collection")
    print("=" * 70)
    print()
    
    all_trials = []
    
    # Collect trials across multiple conditions
    for condition in SEARCH_CONDITIONS:
        trials = fetch_trials(condition, max_results=300)
        all_trials.extend(trials)
        
        # Brief pause between requests to be respectful of API
        time.sleep(API_DELAY)
    
    # Convert to DataFrame
    df = pd.DataFrame(all_trials)
    
    if df.empty:
        print("\nNo trials downloaded. Check your internet connection.")
        return
    
    # Remove any duplicate trials
    original_count = len(df)
    df = df.drop_duplicates(subset=['nct_id'])
    duplicates_removed = original_count - len(df)
    
    # Display summary statistics
    print(f"\n{'=' * 70}")
    print("Data Collection Summary")
    print("=" * 70)
    print(f"Total trials collected: {len(df)}")
    print(f"Duplicates removed: {duplicates_removed}")
    print(f"\nOutcome Distribution:")
    print(f"  Successful trials: {(df['outcome']==1).sum()} ({(df['outcome']==1).sum()/len(df)*100:.1f}%)")
    print(f"  Failed trials: {(df['outcome']==0).sum()} ({(df['outcome']==0).sum()/len(df)*100:.1f}%)")
    
    # Save to CSV
    output_file = 'clinical_trials_data.csv'
    df.to_csv(output_file, index=False)
    print(f"\nData saved to: {output_file}")
    
    # Show sample records
    print(f"\n{'=' * 70}")
    print("Sample Records")
    print("=" * 70)
    
    if (df['outcome']==1).any():
        success_sample = df[df['outcome']==1].iloc[0]
        print(f"\nSuccessful Trial Example:")
        print(f"  ID: {success_sample['nct_id']}")
        print(f"  Title: {success_sample['title'][:70]}...")
        print(f"  Status: {success_sample['status']}")
    
    if (df['outcome']==0).any():
        failure_sample = df[df['outcome']==0].iloc[0]
        print(f"\nFailed Trial Example:")
        print(f"  ID: {failure_sample['nct_id']}")
        print(f"  Title: {failure_sample['title'][:70]}...")
        print(f"  Reason: {failure_sample['why_stopped'][:70]}...")
    
    print(f"\n{'=' * 70}")
    print("Data collection complete. Ready for model training.")
    print("=" * 70)


if __name__ == "__main__":
    main()

Clinical Trials Data Collection

Fetching cancer trials... Retrieved 148 trials
Fetching breast cancer trials... Retrieved 147 trials
Fetching lung cancer trials... Retrieved 145 trials
Fetching diabetes trials... Retrieved 200 trials
Fetching type 2 diabetes trials... Retrieved 200 trials
Fetching cardiovascular disease trials... Retrieved 166 trials
Fetching heart failure trials... Retrieved 178 trials
Fetching depression trials... Retrieved 195 trials
Fetching alzheimer disease trials... Retrieved 184 trials
Fetching asthma trials... Retrieved 214 trials

Data Collection Summary
Total trials collected: 1602
Duplicates removed: 175

Outcome Distribution:
  Successful trials: 1387 (86.6%)
  Failed trials: 215 (13.4%)

Data saved to: clinical_trials_data.csv

Sample Records

Successful Trial Example:
  ID: NCT07375407
  Title: Aerobic Exercise vs Physiological Ischemic Training in Stage 2 Hyperte...
  Status: COMPLETED

Failed Trial Example:
  ID: NCT03249792
  Title: Study of MK-2118 

In [ ]:
"""
Clinical Trial Outcome Prediction Model Training

Fine-tunes BioBERT on clinical trial data to predict trial success/failure.
Uses GPU acceleration (CUDA) when available and handles class imbalance.

Author: [Your Name]
Date: February 2025
"""

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support, 
    roc_auc_score, 
    confusion_matrix, 
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    TrainerCallback
)
from datasets import Dataset
import warnings
import time

warnings.filterwarnings('ignore')

# Model configuration
BASE_MODEL = "dmis-lab/biobert-v1.1"  # Biomedical domain-adapted BERT
MAX_SEQUENCE_LENGTH = 512
RANDOM_SEED = 42

# Training hyperparameters
EPOCHS = 8  # Increased for better convergence
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01


class ProgressCallback(TrainerCallback):
    """Custom callback to display detailed training progress."""
    
    def __init__(self, num_epochs):
        self.num_epochs = num_epochs
        self.current_epoch = 0
        self.epoch_start_time = None
    
    def on_epoch_begin(self, args, state, control, **kwargs):
        """Called at the beginning of each epoch."""
        self.current_epoch = state.epoch
        self.epoch_start_time = time.time()
        print(f"\n{'─' * 70}")
        print(f"Epoch {int(self.current_epoch)}/{self.num_epochs}")
        print(f"{'─' * 70}")
    
    def on_epoch_end(self, args, state, control, **kwargs):
        """Called at the end of each epoch."""
        epoch_time = time.time() - self.epoch_start_time
        print(f"Epoch {int(self.current_epoch)} completed in {epoch_time:.1f}s")
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        """Called when logging training metrics."""
        if logs:
            # Display loss during training
            if 'loss' in logs:
                print(f"  Step {state.global_step}: loss = {logs['loss']:.4f}", end='\r')


def setup_device():
    """Configure and report on compute device (GPU/CPU)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    print("=" * 70)
    print("Clinical Trial Outcome Predictor - Model Training")
    print("=" * 70)
    print(f"Compute Device: {device.upper()}")
    
    if device == "cuda":
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {gpu_name}")
        print(f"GPU Memory: {gpu_memory:.1f} GB")
    else:
        print("Note: Running on CPU. Training will be slower than GPU.")
    
    print("=" * 70)
    return device


def load_and_prepare_data(filepath: str):
    """
    Load trial data and prepare for training.
    
    Args:
        filepath: Path to CSV file with trial data
        
    Returns:
        Prepared DataFrame with combined text features
    """
    print("\nLoading dataset...")
    df = pd.read_csv(filepath)
    print(f"  Loaded {len(df)} trials")
    
    # Handle missing values
    for col in ['description', 'eligibility', 'title']:
        df[col] = df[col].fillna('')
    
    # Combine text fields using special separator token
    # BioBERT uses [SEP] to separate different text segments
    print("Preparing text features...")
    df['full_text'] = (
        df['title'].astype(str) + ' [SEP] ' + 
        df['description'].astype(str) + ' [SEP] ' + 
        df['eligibility'].astype(str)
    )
    
    return df


def analyze_class_distribution(df):
    """Display and calculate class balance information."""
    n_success = (df['outcome'] == 1).sum()
    n_failed = (df['outcome'] == 0).sum()
    total = len(df)
    
    print("\nClass Distribution:")
    print(f"  Successful trials: {n_success:4d} ({n_success/total*100:.1f}%)")
    print(f"  Failed trials:     {n_failed:4d} ({n_failed/total*100:.1f}%)")
    
    # Calculate class weights for handling imbalance
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=df['outcome']
    )
    
    print(f"\nComputed Class Weights (for loss function):")
    print(f"  Failed trials:     {class_weights[0]:.2f}")
    print(f"  Successful trials: {class_weights[1]:.2f}")
    
    return class_weights


def create_train_val_test_splits(df, test_size=0.2, val_size=0.125):
    """
    Split data into train, validation, and test sets with stratification.
    
    Stratification ensures each split has similar class distributions.
    
    Args:
        df: DataFrame with trial data
        test_size: Proportion of data for test set
        val_size: Proportion of training data for validation set
        
    Returns:
        Tuple of (train_df, val_df, test_df)
    """
    print("\nSplitting dataset (stratified by outcome)...")
    
    # First split: separate test set
    train_df, test_df = train_test_split(
        df, 
        test_size=test_size, 
        random_state=RANDOM_SEED, 
        stratify=df['outcome']
    )
    
    # Second split: separate validation from training
    train_df, val_df = train_test_split(
        train_df, 
        test_size=val_size,
        random_state=RANDOM_SEED, 
        stratify=train_df['outcome']
    )
    
    # Report split sizes
    print(f"  Training set:   {len(train_df):4d} trials")
    print(f"    Success: {(train_df['outcome']==1).sum()}, Failed: {(train_df['outcome']==0).sum()}")
    print(f"  Validation set: {len(val_df):4d} trials")
    print(f"    Success: {(val_df['outcome']==1).sum()}, Failed: {(val_df['outcome']==0).sum()}")
    print(f"  Test set:       {len(test_df):4d} trials")
    print(f"    Success: {(test_df['outcome']==1).sum()}, Failed: {(test_df['outcome']==0).sum()}")
    
    return train_df, val_df, test_df


def prepare_dataset_for_training(df_split, tokenizer, dataset_name="dataset"):
    """
    Convert DataFrame to tokenized HuggingFace Dataset.
    
    Args:
        df_split: DataFrame split (train/val/test)
        tokenizer: BioBERT tokenizer
        dataset_name: Name for logging
        
    Returns:
        Prepared HuggingFace Dataset ready for training
    """
    # Convert to HuggingFace Dataset
    dataset = Dataset.from_pandas(
        df_split[['full_text', 'outcome']].reset_index(drop=True)
    )
    
    # Tokenization function
    def tokenize_function(examples):
        return tokenizer(
            examples['full_text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_SEQUENCE_LENGTH
        )
    
    # Apply tokenization
    dataset = dataset.map(tokenize_function, batched=True)
    
    # Rename outcome to labels (required by Trainer)
    dataset = dataset.rename_column('outcome', 'labels')
    
    # Set format for PyTorch
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    return dataset


def compute_metrics(eval_pred):
    """
    Calculate comprehensive evaluation metrics.
    
    Computes accuracy, precision, recall, F1, and ROC-AUC.
    These metrics provide a complete picture of model performance.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate probability scores for AUC
    probs = torch.nn.functional.softmax(
        torch.tensor(eval_pred.predictions), dim=1
    )[:, 1].numpy()
    
    # Compute standard classification metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, predictions)
    
    # ROC-AUC measures discrimination ability
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        # Can fail if only one class present in batch
        auc = 0.0
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }


class WeightedTrainer(Trainer):
    """
    Custom Trainer that applies class weights to the loss function.
    
    This helps the model pay more attention to the minority class (failed trials)
    and improves performance on imbalanced datasets.
    """
    
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Override loss computation to apply class weights."""
        # Extract labels - handle both dict and direct access
        if isinstance(inputs, dict):
            labels = inputs.get("labels")
            if labels is None:
                # Fallback: labels might be passed differently
                labels = inputs.pop("labels", None)
        else:
            labels = inputs.pop("labels")
        
        if labels is None:
            raise KeyError("Labels not found in inputs")
        
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Weighted cross-entropy loss
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=torch.tensor(
                self.class_weights, 
                dtype=torch.float32
            ).to(logits.device)
        )
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss


def display_results(predictions, labels, test_df):
    """Display comprehensive evaluation results and example predictions."""
    preds = np.argmax(predictions.predictions, axis=1)
    probs = torch.nn.functional.softmax(
        torch.tensor(predictions.predictions), dim=1
    )[:, 1].numpy()
    
    # Calculate metrics
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    auc = roc_auc_score(labels, probs)
    
    print("\n" + "=" * 70)
    print("Final Model Performance on Test Set")
    print("=" * 70)
    print(f"\nOverall Metrics:")
    print(f"  Accuracy:  {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1 Score:  {f1:.3f}")
    print(f"  ROC-AUC:   {auc:.3f}")
    
    # Confusion matrix provides detailed breakdown
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\nConfusion Matrix:")
    print(f"                     Predicted")
    print(f"                 Failed  Success")
    print(f"Actual Failed    {cm[0,0]:4d}    {cm[0,1]:4d}")
    print(f"       Success   {cm[1,0]:4d}    {cm[1,1]:4d}")
    
    print(f"\nDetailed Breakdown:")
    print(f"  True Positives (correctly identified failures):  {tp}")
    print(f"  True Negatives (correctly identified successes): {tn}")
    print(f"  False Positives (false alarms):                  {fp}")
    print(f"  False Negatives (missed failures):               {fn}")
    
    # Per-class report
    print(f"\nClassification Report:")
    print(classification_report(
        labels, preds, 
        target_names=['Failed', 'Success'], 
        digits=3
    ))
    
    # Save detailed predictions
    results_df = test_df.copy()
    results_df['predicted'] = preds
    results_df['success_probability'] = probs
    results_df['correct'] = (labels == preds)
    results_df.to_csv('test_predictions.csv', index=False)
    
    # Show interesting examples
    display_example_predictions(results_df)


def display_example_predictions(results_df):
    """Show examples of model predictions for interpretation."""
    print("\n" + "=" * 70)
    print("Example Predictions")
    print("=" * 70)
    
    # High confidence correct prediction
    correct_high_conf = results_df[
        results_df['correct'] & 
        (results_df['success_probability'] > 0.9)
    ]
    
    if len(correct_high_conf) > 0:
        sample = correct_high_conf.iloc[0]
        print(f"\nHigh Confidence Correct Prediction:")
        print(f"  Trial: {sample['title'][:65]}...")
        print(f"  Actual: {'SUCCESS' if sample['outcome']==1 else 'FAILED'}")
        print(f"  Predicted: {'SUCCESS' if sample['predicted']==1 else 'FAILED'}")
        print(f"  Confidence: {sample['success_probability']*100:.1f}%")
    
    # Model error for analysis
    errors = results_df[~results_df['correct']]
    if len(errors) > 0:
        sample = errors.iloc[0]
        print(f"\nModel Error (for learning):")
        print(f"  Trial: {sample['title'][:65]}...")
        print(f"  Actual: {'SUCCESS' if sample['outcome']==1 else 'FAILED'}")
        print(f"  Predicted: {'SUCCESS' if sample['predicted']==1 else 'FAILED'}")
        print(f"  Confidence: {sample['success_probability']*100:.1f}%")
        if sample['why_stopped']:
            print(f"  Termination reason: {sample['why_stopped'][:60]}...")


def main():
    """Main training pipeline."""
    # Setup
    device = setup_device()
    
    # Load data
    df = load_and_prepare_data('clinical_trials_data.csv')
    class_weights = analyze_class_distribution(df)
    
    # Create splits
    train_df, val_df, test_df = create_train_val_test_splits(df)
    
    # Load model and tokenizer
    print(f"\nLoading base model: {BASE_MODEL}")
    print("Note: This is a pre-trained model that will be fine-tuned on your data")
    
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        problem_type="single_label_classification"
    )
    model.to(device)
    
    # Prepare datasets
    print("\nPreparing datasets for training...")
    train_dataset = prepare_dataset_for_training(train_df, tokenizer, "training")
    val_dataset = prepare_dataset_for_training(val_df, tokenizer, "validation")
    test_dataset = prepare_dataset_for_training(test_df, tokenizer, "test")
    print("  Datasets prepared successfully")
    
    # Configure training
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        learning_rate=LEARNING_RATE,
        logging_dir='./logs',
        logging_steps=20,  # Log every 20 steps
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        fp16=True,  # Mixed precision for faster training on modern GPUs
        dataloader_num_workers=4,
        report_to="none",  # Disable wandb/tensorboard reporting
    )
    
    # Create progress callback
    progress_callback = ProgressCallback(num_epochs=EPOCHS)
    
    # Create trainer
    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[progress_callback],
    )
    
    # Train
    print("\n" + "=" * 70)
    print("Training Model")
    print("=" * 70)
    print(f"Total epochs: {EPOCHS}")
    print(f"Estimated time: 8-15 minutes on RTX 4060 Ti")
    print()
    
    start_time = time.time()
    trainer.train()
    training_time = (time.time() - start_time) / 60
    
    print(f"\n{'=' * 70}")
    print(f"Training completed in {training_time:.1f} minutes")
    print(f"{'=' * 70}")
    
    # Evaluate
    print("\n" + "=" * 70)
    print("Evaluating on Test Set")
    print("=" * 70)
    
    predictions = trainer.predict(test_dataset)
    display_results(predictions, predictions.label_ids, test_df)
    
    # Save model
    model_path = './final_model'
    print(f"\nSaving trained model to: {model_path}")
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    
    print("\n" + "=" * 70)
    print("Training Complete")
    print("=" * 70)
    print(f"Model saved to: {model_path}")
    print(f"Predictions saved to: test_predictions.csv")
    print("=" * 70)


if __name__ == "__main__":
    main()

c:\Users\Guhan Venkat\AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Clinical Trial Outcome Predictor - Model Training
Compute Device: CUDA
GPU: NVIDIA GeForce RTX 5060 Ti
GPU Memory: 17.1 GB

Loading dataset...
  Loaded 1602 trials
Preparing text features...

Class Distribution:
  Successful trials: 1387 (86.6%)
  Failed trials:      215 (13.4%)

Computed Class Weights (for loss function):
  Failed trials:     3.73
  Successful trials: 0.58

Splitting dataset (stratified by outcome)...
  Training set:   1120 trials
    Success: 970, Failed: 150
  Validation set:  161 trials
    Success: 139, Failed: 22
  Test set:        321 trials
    Success: 278, Failed: 43

Loading base model: dmis-lab/biobert-v1.1
Note: This is a pre-trained model that will be fine-tuned on your data


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Preparing datasets for training...


Map: 100%|██████████| 321/321 [00:00<00:00, 4367.45 examples/s]


  Datasets prepared successfully

Training Model
Total epochs: 8
Estimated time: 8-15 minutes on RTX 4060 Ti


──────────────────────────────────────────────────────────────────────
Epoch 0/8
──────────────────────────────────────────────────────────────────────


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
1,0.712500,0.625843,0.757764,0.851711,0.903226,0.805755,0.749182
2,0.616400,0.600563,0.677019,0.786885,0.914286,0.690647,0.723676
3,0.532600,0.642442,0.857143,0.920415,0.886667,0.956835,0.741171
4,0.430100,0.642765,0.788820,0.872180,0.913386,0.834532,0.753107
5,0.222500,0.972245,0.819876,0.897527,0.881944,0.913669,0.732505
6,0.129900,1.557112,0.863354,0.925676,0.872611,0.985612,0.694081
7,0.060500,1.714762,0.838509,0.910959,0.869281,0.956835,0.689993
8,0.038200,1.891050,0.844720,0.914676,0.870130,0.964029,0.694081


Epoch 0 completed in 25.8s

──────────────────────────────────────────────────────────────────────
Epoch 1/8
──────────────────────────────────────────────────────────────────────
Epoch 1 completed in 25.0s

──────────────────────────────────────────────────────────────────────
Epoch 2/8
──────────────────────────────────────────────────────────────────────
Epoch 2 completed in 25.0s

──────────────────────────────────────────────────────────────────────
Epoch 3/8
──────────────────────────────────────────────────────────────────────
Epoch 3 completed in 25.2s

──────────────────────────────────────────────────────────────────────
Epoch 4/8
──────────────────────────────────────────────────────────────────────
Epoch 4 completed in 24.7s

──────────────────────────────────────────────────────────────────────
Epoch 5/8
──────────────────────────────────────────────────────────────────────
Epoch 5 completed in 24.7s

──────────────────────────────────────────────────────────────────────
E


Final Model Performance on Test Set

Overall Metrics:
  Accuracy:  0.844 (84.4%)
  Precision: 0.863
  Recall:    0.975
  F1 Score:  0.916
  ROC-AUC:   0.592

Confusion Matrix:
                     Predicted
                 Failed  Success
Actual Failed       0      43
       Success      7     271

Detailed Breakdown:
  True Positives (correctly identified failures):  271
  True Negatives (correctly identified successes): 0
  False Positives (false alarms):                  43
  False Negatives (missed failures):               7

Classification Report:
              precision    recall  f1-score   support

      Failed      0.000     0.000     0.000        43
     Success      0.863     0.975     0.916       278

    accuracy                          0.844       321
   macro avg      0.432     0.487     0.458       321
weighted avg      0.747     0.844     0.793       321


Example Predictions

High Confidence Correct Prediction:
  Trial: Safety/Efficacy of Sitagliptin in Patient w/ 